# B1.2 · What the loop may touch: tools, depth and doing it twice

**Function B — Application Security with an AI SDLC → The Agentic Harness**  ·  *Both directions*

Builds on **[B1.1 · The loop, and the verifier that decides what it may conclude](https://spbreed.github.io/cyber-commons/lessons/B1.1.html)**.

| | |
|---|---|
| Tools used | JSON Schema |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A tool's signature decides what the model is able to ask for, which is a stronger control than anything in a prompt. Two tools with identical capability differ entirely when the guard underneath them has a bug: one presents the vulnerable surface, the other cannot express it.

> **At CyberTravels.** `read_voucher(path)` lets the File System Agent ask for any file on the box; `read_voucher(booking_ref)` cannot express the question. And the Workflow Agent issues refunds, so doing it twice is R1, not noise.

## 2 · The framework

```
   three bounds, none of them the prompt

   1  signature      read_voucher(path: str)
                        -> the model may ask for any path
                     read_voucher(ref: Literal[...])
                        -> the surface is not expressible

   2  depth          traveller -> workflow -> sub -> sub -> sub
                     each hop may widen authority; by hop 4
                     nobody has seen the whole path
                     enforce at the ISSUER, not the orchestrator

   3  idempotency    set a config twice  = once
                     post a comment twice = noise
                     issue a REFUND twice = an incident
```

The loop is bounded by three things, and none of them is the prompt.

**What a tool's signature lets the model ask for.** Tool design is security
design, and it is stronger than anything you can put in a prompt, because a
signature decides what is *expressible*:

```python
read_voucher(path: str)                    # any path on the box
read_voucher(booking_ref: Literal[...])    # one of five bookings
```

Both may be safe if a path guard sits underneath. The difference appears when
the guard has a bug: the first tool *presents* the vulnerable surface, the
second never expresses it. Three rules follow, and they compose — enumerate
rather than accept free text, take the narrowest type that works, and return the
least that satisfies the caller.

**How far it may delegate.** Sub-agents buy specialisation. What they also buy,
and what is never on the roadmap, is **delegation depth**. Two things grow with
it: authority composition, since each hop is a place authority could widen if
the narrowing rules from A2.3 are not enforced at *every* hop; and attribution
distance, since by hop four the action is five identities from the human who
asked and no single reviewer has seen the whole path. The control is a depth
limit, and the interesting question is where to enforce it — the orchestrator is
the obvious choice and the wrong one, because when the orchestrator is the
compromised component its own check is worth nothing.

**Whether doing it twice matters.** Your agent will repeat an action. Retries,
restarts, a duplicated webhook, a loop that lost track — the cause varies and
the outcome does not. Whether it matters depends entirely on the action: setting
a config value twice is setting it once; posting a comment twice is noise;
**issuing a refund twice is an incident**, and the Workflow Agent issues
refunds. The mechanism is an idempotency key derived from the *intent*, and the
subtlety is what goes into it. Include a timestamp and every retry is a new
operation.

The second half of that is **replay**, the same property seen from forensics: a
run you cannot deterministically replay is a run you can describe but not
demonstrate, and D2.5 depends on this being right.

## 3 · Demo — the same capability, three signatures

In [ ]:
import fnmatch
from typing import Literal

DOCS = {"runbook": "/srv/docs/runbook.md", "policy": "/srv/docs/policy.md",
        "oncall":  "/srv/docs/oncall.md"}
FILESYSTEM = {**{p: f"contents of {k}" for k, p in DOCS.items()},
              "/home/app/.aws/credentials": "AKIA…SECRET",
              "/etc/shadow": "root:$6$…"}

def normalise(p):
    parts = []
    for seg in p.split("/"):
        if seg in ("", "."): continue
        if seg == "..":
            if parts: parts.pop()
            continue
        parts.append(seg)
    return "/" + "/".join(parts)

def guard(path, workspace="/srv/docs"):
    real = normalise(path)
    if fnmatch.fnmatch(real, "*/.aws/*") or fnmatch.fnmatch(real, "*/etc/shadow"):
        return False
    return real.startswith(workspace + "/")

# --- signature A: free-form path -------------------------------------
def read_file_freeform(path: str):
    if not guard(path):
        return {"error": "denied by path guard"}
    return {"content": FILESYSTEM.get(normalise(path), "not found")}

# --- signature B: enumerated document id ------------------------------
def read_file_enumerated(doc_id: str):
    if doc_id not in DOCS:
        return {"error": f"unknown doc_id; valid: {sorted(DOCS)}"}
    return {"content": FILESYSTEM[DOCS[doc_id]]}

REQUESTS = ["runbook", "/srv/docs/runbook.md",
            "/srv/docs/../../home/app/.aws/credentials", "/etc/shadow"]
print(f"{'request':46s}{'free-form':28s}enumerated")
print("-" * 92)
for r in REQUESTS:
    a = read_file_freeform(r) if r.startswith("/") else {"error": "not a path"}
    b = read_file_enumerated(r)
    print(f"{r:46s}{str(a)[:26]:28s}{str(b)[:34]}")

## 4 · Where it breaks — introduce one bug in the guard

Both signatures were safe above, because the guard worked. Now make the guard wrong in the ordinary way (A3.3's bug: prefix check before normalisation) and re-run. Only one signature is affected.

In [ ]:
def guard_buggy(path, workspace="/srv/docs"):
    return path.startswith(workspace)          # the classic bug

def read_file_freeform_buggy(path: str):
    if not guard_buggy(path):
        return {"error": "denied"}
    return {"content": FILESYSTEM.get(normalise(path), "not found")}

attack = "/srv/docs/../../home/app/.aws/credentials"
print("with a buggy path guard:")
print(f"   free-form  : {read_file_freeform_buggy(attack)}")
print(f"   enumerated : {read_file_enumerated(attack)}")
print("\nThe enumerated tool is unaffected by a filesystem bug it never touches.")
print("It cannot express the request, so the guard's correctness stops mattering.")

In [ ]:
USER_RECORD = {"id": 4471, "email": "dana@corp", "name": "Dana",
               "ssn": "123-45-6789", "salary": 145000,
               "mfa_secret": "JBSWY3DPEHPK3PXP", "role": "engineer"}

def get_user_wide(user_id):            return USER_RECORD
def get_user_narrow(user_id, fields):
    allowed = {"id", "name", "email", "role"}
    bad = set(fields) - allowed
    if bad:
        return {"error": f"fields not exposed by this tool: {sorted(bad)}"}
    return {k: USER_RECORD[k] for k in fields}

print("wide tool returns  :", sorted(get_user_wide(4471)))
print("narrow, legitimate :", get_user_narrow(4471, ["name", "role"]))
print("narrow, overreach  :", get_user_narrow(4471, ["name", "ssn", "mfa_secret"]))

leaked = set(get_user_wide(4471)) & {"ssn", "mfa_secret", "salary"}
print(f"\nsensitive fields placed in the model's context by the wide tool: {sorted(leaked)}")
assert not (set(get_user_narrow(4471, ["name", "role"])) & leaked)

In [ ]:
# Verify: score a tool signature against the three rules.
def review_signature(name, accepts_free_text, bounded_types, returns_minimum):
    score = sum([not accepts_free_text, bounded_types, returns_minimum])
    problems = []
    if accepts_free_text:
        problems.append("accepts free text where an enumeration would do")
    if not bounded_types:
        problems.append("unbounded types — parse errors become the guard's problem")
    if not returns_minimum:
        problems.append("returns more than the caller needs")
    return {"tool": name, "score": f"{score}/3", "problems": problems}

TOOLS = [
 ("read_file(path: str)",                   True,  False, True),
 ("read_file(doc_id: Literal[...])",        False, True,  True),
 ("get_user(user_id: int)",                 False, True,  False),
 ("get_user(user_id: int, fields: list)",   False, True,  True),
 ("run_shell(cmd: str)",                    True,  False, False),
]
for name, free, bounded, minimal in TOOLS:
    r = review_signature(name, free, bounded, minimal)
    print(f"{r['score']}  {r['tool']}")
    for p in r["problems"]:
        print(f"        ⚠ {p}")

## 5 · Depth — every hop is a place authority can widen

A tool signature bounds one call. Sub-agents multiply the calls, and delegation depth is the axis nobody puts on the roadmap.

In [ ]:
from dataclasses import dataclass, field

CEILINGS = {"dana@corp": {"repo:read","repo:write","deploy:prod"},
            "orchestrator": {"repo:read","repo:write","deploy:prod"},
            "planner":   {"repo:read"},
            "coder":     {"repo:read","repo:write"},
            "reviewer":  {"repo:read"},
            "shipper":   {"repo:read","deploy:prod"}}

class DelegationError(Exception): pass

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c
    @property
    def depth(self): return len(self.chain())

def exchange(pres, actor, scopes):
    scopes = set(scopes)
    if not scopes <= pres.scopes:
        raise DelegationError(f"widening: {sorted(scopes - pres.scopes)}")
    if not scopes <= CEILINGS.get(actor, set()):
        raise DelegationError(f"above {actor}'s ceiling")
    return Token(pres.sub, actor, scopes, {"actor": pres.actor, "act": pres.act})

root  = Token("dana@corp", "dana@corp", set(CEILINGS["dana@corp"]))
orch  = exchange(root, "orchestrator", {"repo:read","repo:write","deploy:prod"})
coder = exchange(orch, "coder", {"repo:read","repo:write"})
rev   = exchange(coder, "reviewer", {"repo:read"})

for t in (root, orch, coder, rev):
    print(f"depth {t.depth}  {' → '.join(t.chain()):52s} {sorted(t.scopes)}")

## 6 · Where it breaks — the check in the wrong place

The orchestrator enforces `MAX_DEPTH`. Now assume the orchestrator is the compromised component, which is the realistic case: it is the one processing untrusted task descriptions.

In [ ]:
MAX_DEPTH = 3

def orchestrator_enforced(chain_token, new_actor, scopes, honest=True):
    """The limit lives inside the orchestrator's own code."""
    if honest and chain_token.depth + 1 > MAX_DEPTH:
        raise DelegationError(f"depth {chain_token.depth+1} > {MAX_DEPTH}")
    return exchange(chain_token, new_actor, scopes)

print("honest orchestrator:")
try:
    t = orchestrator_enforced(rev, "shipper", {"repo:read"}, honest=True)
    print("   granted:", t.chain())
except DelegationError as e:
    print("   refused:", e)

print("\ncompromised orchestrator (simply does not run its own check):")
t = orchestrator_enforced(rev, "shipper", {"repo:read"}, honest=False)
print(f"   granted: depth {t.depth}  {' → '.join(t.chain())}")
print("   The limit was a line of code inside the component we no longer trust.")

## 7 · The control — enforce at the issuer and the resource server

Both of these are outside the orchestrator's control, so a compromised orchestrator cannot skip them.

In [ ]:
def issuer_exchange(pres, actor, scopes, max_depth=MAX_DEPTH):
    """The token issuer counts the act chain it is being asked to extend."""
    if pres.depth + 1 > max_depth:
        raise DelegationError(f"issuer refuses: depth {pres.depth+1} > {max_depth}")
    return exchange(pres, actor, scopes)

def resource_server(token, scope, max_depth=MAX_DEPTH):
    """Independent second check, at the point the action actually happens."""
    if token.depth > max_depth:
        return False, f"resource server refuses: depth {token.depth} > {max_depth}"
    if scope not in token.scopes:
        return False, f"missing scope {scope}"
    return True, f"allowed for {' → '.join(token.chain())}"

print("issuer enforcement:")
try:
    issuer_exchange(rev, "shipper", {"repo:read"})
except DelegationError as e:
    print("   ", e)

print("\nresource server enforcement (even if a deep token somehow exists):")
deep = Token("dana@corp", "shipper", {"repo:read"},
             {"actor":"reviewer","act":{"actor":"coder",
              "act":{"actor":"orchestrator","act":None}}})
print(f"   depth {deep.depth}: {resource_server(deep, 'repo:read')}")
print(f"   depth {rev.depth}: {resource_server(rev, 'repo:read')}")

In [ ]:
# Verify: no chain the compromised orchestrator can build is ever honoured.
import random
random.seed(9)
actors = ["planner","coder","reviewer","shipper"]
honoured_too_deep = 0
for _ in range(3000):
    tok = Token("dana@corp","dana@corp", set(CEILINGS["dana@corp"]))
    for _ in range(random.randint(1,6)):
        a = random.choice(actors)
        want = set(random.sample(sorted(tok.scopes),
                                 k=random.randint(0,len(tok.scopes))))
        try:
            tok = exchange(tok, a, want)      # compromised: no depth check
        except DelegationError:
            break
    ok, _ = resource_server(tok, "repo:read")
    if ok and tok.depth > MAX_DEPTH:
        honoured_too_deep += 1
print(f"3000 chains built without any orchestrator-side limit — "
      f"over-deep chains honoured by the resource server: {honoured_too_deep}")
assert honoured_too_deep == 0
print("The limit holds because it is enforced where the orchestrator cannot reach.")

## 8 · And it will all happen twice

Retries, restarts, a duplicated webhook, a loop that lost track. Whether that matters depends entirely on the action.

In [ ]:
import hashlib, json
from dataclasses import dataclass, field

@dataclass
class Ledger:
    applied: dict = field(default_factory=dict)
    effects: list = field(default_factory=list)

    def apply(self, key, op, **args):
        if key in self.applied:
            self.effects.append(f"SKIP  {op} (key {key[:8]} already applied)")
            return False
        self.applied[key] = (op, args)
        self.effects.append(f"APPLY {op} {args}")
        return True

def idem_key(op, **args):
    """Derived from INTENT. No timestamp, no attempt number."""
    blob = json.dumps({"op": op, "args": args}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()

led = Ledger()
# the agent retries the same refund three times
for attempt in range(3):
    k = idem_key("issue_refund", order="ORD-4471", amount=250)
    led.apply(k, "issue_refund", order="ORD-4471", amount=250)
# a genuinely different refund
led.apply(idem_key("issue_refund", order="ORD-4472", amount=90),
          "issue_refund", order="ORD-4472", amount=90)

print("\n".join(led.effects))
print(f"\n4 calls → {len(led.applied)} effects")

## 9 · Where it breaks — a key that includes the wrong thing

In [ ]:
import time

def bad_key_timestamp(op, **args):
    return hashlib.sha256(f"{op}{args}{time.time()}".encode()).hexdigest()

def bad_key_too_narrow(op, **args):
    return hashlib.sha256(op.encode()).hexdigest()          # op only!

led2 = Ledger()
for attempt in range(3):
    led2.apply(bad_key_timestamp("issue_refund", order="ORD-4471", amount=250),
               "issue_refund", order="ORD-4471", amount=250)
print("key includes a timestamp — every retry is a new operation:")
print("\n".join(led2.effects))
print(f"→ refunded {sum(1 for e in led2.effects if e.startswith('APPLY')) * 250} "
      f"instead of 250\n")

led3 = Ledger()
led3.apply(bad_key_too_narrow("issue_refund", order="ORD-4471", amount=250),
           "issue_refund", order="ORD-4471", amount=250)
led3.apply(bad_key_too_narrow("issue_refund", order="ORD-9999", amount=800),
           "issue_refund", order="ORD-9999", amount=800)
print("key includes only the op — different refunds collide:")
print("\n".join(led3.effects))
print("→ the second customer never got their money")

## 10 · The control — classify the action, then key on intent

In [ ]:
ACTIONS = {
 "set_config":     ("naturally idempotent", False),
 "add_label":      ("naturally idempotent", False),
 "post_comment":   ("accumulating",         True),
 "send_email":     ("accumulating",         True),
 "issue_refund":   ("dangerous",            True),
 "rotate_secret":  ("dangerous",            True),
 "scale_cluster":  ("dangerous",            True),
}
print(f"{'action':16s}{'nature':22s}needs a key")
print("-" * 52)
for a, (nature, needs) in ACTIONS.items():
    print(f"{a:16s}{nature:22s}{needs}")

def guarded_call(led, op, **args):
    nature, needs_key = ACTIONS[op]
    if not needs_key:
        led.effects.append(f"APPLY {op} {args} (idempotent by nature)")
        return True
    return led.apply(idem_key(op, **args), op, **args)

led4 = Ledger()
for _ in range(2):
    guarded_call(led4, "set_config", key="tls_min", value="1.2")
    guarded_call(led4, "issue_refund", order="ORD-4471", amount=250)
print("\n" + "\n".join(led4.effects))

In [ ]:
# Verify — replay: the forensic half of the same property.
@dataclass
class Replay:
    prompts: list = field(default_factory=list)
    tool_results: list = field(default_factory=list)
    model_version: str = ""
    seed: object = None
    def replayable(self):
        missing = []
        if not self.prompts:      missing.append("prompts not recorded")
        if not self.tool_results: missing.append("tool results not recorded — "
                                                 "the agent saw a world you cannot rebuild")
        if not self.model_version: missing.append("model version not pinned — "
                                                  "a silent upgrade changes the output")
        if self.seed is None:     missing.append("no seed — sampling makes it unrepeatable")
        return (not missing), missing

for name, r in (("fully instrumented", Replay(["p"], ["tool out"], "glm-4.6@2026-07", 42)),
                ("typical production", Replay(["p"], ["tool out"], "", None)),
                ("actions only",       Replay([], [], "", None))):
    ok, missing = r.replayable()
    print(f"{name:22s} replayable={ok}")
    for m in missing: print(f"      ✗ {m}")
assert Replay(["p"], ["t"], "m", 1).replayable()[0]

## What you just proved

The same capability behind three signatures: the free-text one presents a traversal surface the enumerated one cannot express, and introducing one bug in the shared guard breaks only the tools that expose the argument. A depth-3 delegation chain narrows correctly, then the same chain widens when the check lives in the compromised orchestrator rather than at the issuer. Finally the same action is classified three ways, and an idempotency key containing a timestamp makes every retry a new refund.

## Your turn

Take one tool your agent can call and write its signature as a type. If a parameter is `str` where the real requirement is a choice from a known set, narrowing it costs an afternoon and removes a class of bug permanently. Then check where your delegation-depth limit is enforced — if it is in the orchestrator, it protects you from everything except the orchestrator.

---

**Next → [B1.3 · Which model runs it, and who checks the checker](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*